# 10 — Desarrollo de modelos de referencia

Prueba de ingeniería sobre MIMIC-IV Demo v2.2 para Sepsis-3 a 6 horas. Solo se abren `development` y `validation`; `test` no se descubre ni se carga. El tamaño y el número de eventos del demo impiden interpretar las métricas como rendimiento clínico.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, tempfile
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.modeling import (assemble_modeling_table, binary_metrics, equal_patient_weights, grouped_cross_validation, grouped_prevalence_cross_validation, make_logistic_pipeline, patient_weighted_event_rate)
from scripts.build_demo_sofa_incremental import canonical_config
artifact_config = canonical_config()
model_config = json.loads((PROJECT_ROOT/'config/modeling.json').read_text())
candidates = sorted((PROJECT_ROOT/'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json'))
valid = []
for path in candidates:
    try:
        manifest = ArtifactStore(path.parent).validate('sepsis3_development_features', expected_config=artifact_config)
        valid.append((manifest.created_at_utc, path.parents[1]))
    except (FileNotFoundError, ArtifactValidationError):
        pass
if not valid: raise RuntimeError('Ejecute primero los notebooks 00–09 o el pipeline incremental.')
RUN_ROOT = sorted(valid, key=lambda item: (item[0], str(item[1])))[-1][1]
landmarks = ArtifactStore(RUN_ROOT/'50_landmarks'); features = ArtifactStore(RUN_ROOT/'60_features')
print(f'Ejecución validada: {RUN_ROOT.name}')

## Ensamblaje del horizonte primario

In [ ]:
def load_partition(partition):
    return assemble_modeling_table(
        landmarks.read_dataframe(f'sepsis3_{partition}_landmarks', expected_config=artifact_config),
        features.read_dataframe(f'sepsis3_{partition}_features', expected_config=artifact_config),
        horizon_hours=model_config['primary_horizon_hours'])
development = load_partition('development')
validation = load_partition('validation')
audit = pd.DataFrame([
    {'partition':'development', 'landmarks':len(development), 'patients':development.subject_id.nunique(), 'stays':development.stay_id.nunique(), 'events':int(development.outcome.sum())},
    {'partition':'validation', 'landmarks':len(validation), 'patients':validation.subject_id.nunique(), 'stays':validation.stay_id.nunique(), 'events':int(validation.outcome.sum())}])
audit['event_percent'] = 100*audit.events/audit.landmarks
display(audit)
if audit.events.min() < 20: print('ADVERTENCIA: menos de 20 eventos; métricas exclusivamente técnicas.')

## Validación agrupada y validación separada

La imputación y el escalado se ajustan dentro de cada fold. Cada paciente recibe el mismo peso total aunque contribuya con distinto número de landmarks.

In [ ]:
columns = model_config['clinical_baseline_features']
folds = model_config['cross_validation_folds']; seed = model_config['seed']
prevalence_oof, prevalence_folds = grouped_prevalence_cross_validation(development, folds=folds, seed=seed)
pipeline = make_logistic_pipeline(columns, c=model_config['logistic_c'], seed=seed)
clinical_oof, clinical_folds = grouped_cross_validation(development, pipeline, columns, folds=folds, seed=seed)
pipeline.fit(development[columns], development.outcome, model__sample_weight=equal_patient_weights(development))
clinical_validation_probability = pipeline.predict_proba(validation[columns])[:,1]
prevalence_probability = patient_weighted_event_rate(development)
metrics = pd.DataFrame([
    {'model':'prevalence', 'sample':'development_oof', **binary_metrics(prevalence_oof.outcome, prevalence_oof.probability)},
    {'model':'clinical_logistic', 'sample':'development_oof', **binary_metrics(clinical_oof.outcome, clinical_oof.probability)},
    {'model':'prevalence', 'sample':'validation', **binary_metrics(validation.outcome, [prevalence_probability]*len(validation))},
    {'model':'clinical_logistic', 'sample':'validation', **binary_metrics(validation.outcome, clinical_validation_probability)}])
display(metrics, clinical_folds)

## Calibración descriptiva con ggplot2

In [ ]:
calibration = pd.DataFrame({'outcome':validation.outcome.to_numpy(), 'probability':clinical_validation_probability})
calibration['bin'] = pd.qcut(calibration.probability, q=5, duplicates='drop')
calibration = calibration.groupby('bin', observed=True).agg(n=('outcome','size'), events=('outcome','sum'), predicted=('probability','mean'), observed=('outcome','mean')).reset_index(drop=True)
display(calibration)
rscript = shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible.')
with tempfile.TemporaryDirectory() as tmp:
    tmp=Path(tmp); csv=tmp/'calibration.csv'; svg=tmp/'calibration.svg'; script=tmp/'plot.R'
    calibration.to_csv(csv,index=False)
    script.write_text("""args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[1])
p <- ggplot(d,aes(predicted,observed,size=n)) + geom_abline(slope=1,intercept=0,linetype=2,colour='grey50') + geom_point(colour='#D9534F') + scale_size_continuous(range=c(2,7)) + labs(title='Calibración descriptiva — validación demo',subtitle='Muy pocos eventos: no usar para inferencia',x='Probabilidad media predicha',y='Proporción observada',size='Landmarks') + theme_minimal(base_size=12)
ggsave(args[2],p,width=7,height=5,device=grDevices::svg)
""")
    result=subprocess.run([rscript,str(script),str(csv),str(svg)],capture_output=True,text=True)
    if result.returncode: raise RuntimeError(result.stderr)
    display(SVG(filename=str(svg)))

## Interpretación

Este notebook valida el circuito estadístico, no un modelo clínico. La rareza y concentración de eventos hacen inestables AUROC, AUPRC y calibración. No se selecciona modelo, umbral ni variable con estos resultados; esas decisiones requieren MIMIC-IV completo y permanecen anteriores a cualquier acceso al test final.